In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
ARIZE_SPACE_ID=os.getenv("ARIZE_SPACE_ID")
ARIZE_API_KEY=os.getenv("ARIZE_API_KEY")
OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")

# Connection with arize

In [ ]:
from arize.otel import register
from openinference.instrumentation.openai import OpenAIInstrumentor
from openinference.instrumentation.agno import AgnoInstrumentor
import httpx

model_id="E-commerce Agent"
tracer_provider=register(
    space_id=os.getenv("ARIZE_SPACE_ID"),
    api_key=os.getenv("ARIZE_API_KEY"),
    project_name=model_id,
    set_global_tracer_provider=True
)
OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)
AgnoInstrumentor().instrument(tracer_provider=tracer_provider)




In [ ]:
from opentelemetry import trace
tracer=trace.get_tracer(__name__)


In [ ]:
def _search_api(query: str)-> str | None:
    """Try tavily search first, fall back to None"""
    tavily_key=os.getenv("TAVILY_API_KEY")
    if not tavily_key:
        return None
    try:
        resp=httpx.post(
            "https://api.tavily.com/search",
            json={
                "api_key":tavily_key,
                "query": query,
                "max_result":3,
                "search_depth":"basic",
                "include_answer":True,
            },
            timeout=8,
        )
        data=resp.json()
        answer=data.get("answer") or ""
        snippets=[r.get("content","") for r in data.get("result",[])]
        combined=" ".join([answer] + snippets).strip()
        return combined[:400] if combined else None
    except Exception:
        return None

def _compact(text: str, limit: int = 200)-> str:
    cleaned=" ".join(text.split())
    if len(cleaned) <= limit:
        return cleaned
    else:
        return cleaned[:limit].rsplit(" ", 1)[0]

In [ ]:
from agno.tools import tool

@tool
def product_search(product: str)->str:
    """Search general info about a product"""
    q=f"{product} features price specification and reviews"
    s=_search_api(q)
    if s:
        return f"{product} info: {_compact(s)}"
    return f"{product} has various models with different features and prices."

@tool
def product_compare(product1: str, product2: str)-> str:
    """Compare 2 product"""
    q=f"{product1} vs {product2} comparison, features, price,  pros and cons"
    s=_search_api(q)
    if s:
        return f"comparison {product1} vs {product2}: {_compact(s)}"
    return f"{product1} and {product2} differ in features, performance and priceing"

@tool 
def budget_finder(product: str, budget: str)-> str:
    """find best option under budget"""
    q=f"best {product} under {budget} recommendation"
    s=_search_api(q)
    if s:
        return f"Best {product} under {budget}: {_compact(s)}"
    return f"you can find good {product} option under {budget} with balanced features."

In [ ]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat

ecommerce_agent=Agent(
    name="SmartShopper",
    role="AI E-commerce Assistant",
    model=OpenAIChat(id="gpt-4o-mini"),
    instructions=(
        "You are a smart shopping assistant"
        "Help user find, compare, and choose the best products"
        "Use tool to provide accurate info"
        "Keep answer clear and helpfull"
    ),
    markdown=True,
    tools=[product_search, product_compare, budget_finder]
)

In [ ]:
product="apple phone"
product1="apple phone"
product2="sumsang phone"
budget="50k"

query=f"""
I want to buy {product} under {budget}
compare {product1} and {product2} and suggest the best option
"""
ecommerce_agent.print_response(
    query,
    stream=True
)
